# 定义评测指标：Correctness / Faithfulness / Retrieval Relevance

这一节的目标很简单：**不用引入第三方评测框架**，仅用 LangChain + 一个 LLM，当作“裁判模型（LLM-as-judge）”，把你关心的评测维度封装成可复用函数。

我们会实现 3 个最常见、也最容易混淆的指标：

- **Correctness（正确性）**：回答 vs 标准答案（需要 ground truth）。输出 0~1。
- **Faithfulness / Groundedness（忠实性/可溯源）**：回答是否能从给定上下文中推出（不关心它在客观世界里对不对）。输出 0/1。
- **Retrieval relevance（检索相关性）**：检索到的 docs 对问题是否有用（这里用“平均相关度”近似）。输出 0~1。

> 这三类也对应 LangChain/LangSmith 文档里对 RAG 评测的常见拆解：回答正确性、回答是否被上下文支撑、以及检索结果是否相关。

## 0) 环境准备

请确保：

- 已设置 `OPENAI_API_KEY`（例如放在 `.env`）
- 已安装：`langchain` / `langchain-core` / `langchain-openai` / `python-dotenv` / `pydantic`

> 说明：这里我们只演示“如何定义指标本身”。把这些指标接到 LangSmith / DeepEval / 你自己的离线评测循环里，是后续的工程化步骤。

In [1]:
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [2]:
# 代理设置（如不需要可自行注释掉）
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "socks5://127.0.0.1:7890"

In [3]:
class ResultScore(BaseModel):
    """一个统一的返回结构：分数 + 解释。"""

    score: float = Field(
        ..., description="0~1 之间的分数（1 表示最好）",
    )
    explanation: str = Field(..., description="给分理由（尽量简洁、指向性强）")

## 1) Correctness：回答是否与标准答案一致？

这类指标需要 ground truth（标准答案）。

- **输入**：`question` / `ground_truth` / `generated_answer`
- **输出**：0~1（越接近 1 越正确）

注意：这里的 correctness 更接近“语义正确/覆盖程度”，不要求逐字一致。

In [4]:
correctness_prompt = PromptTemplate(
    input_variables=["question", "ground_truth", "generated_answer"],
    template="""
Question: {question}
Ground Truth: {ground_truth}
Generated Answer: {generated_answer}

Evaluate the correctness of the generated answer compared to the ground truth.
Score from 0 to 1, where 1 is perfectly correct and 0 is completely incorrect.
Any score between 0 and 1 is acceptable.

Return your result as JSON.
""".strip(),
)

correctness_chain = correctness_prompt | llm.with_structured_output(ResultScore)


def evaluate_correctness(question: str, ground_truth: str, generated_answer: str) -> ResultScore:
    return correctness_chain.invoke(
        {
            "question": question,
            "ground_truth": ground_truth,
            "generated_answer": generated_answer,
        }
    )

In [5]:
# quick test
question = "What is the capital of France and Spain?"
ground_truth = "Paris and Madrid"
generated_answer = "Paris"

result = evaluate_correctness(question, ground_truth, generated_answer)
result

ResultScore(score=0.5, explanation='The generated answer correctly identifies the capital of France as Paris, but it fails to mention the capital of Spain, which is Madrid. Therefore, it is partially correct.')

## 2) Faithfulness / Groundedness：回答能否从上下文中推出？

这里我们只关心：**生成答案是否“由 context 支撑”**。

- 即便答案在现实中是对的，只要 `context` 没给出支撑，就应判为不忠实。
- 这类指标常用于检测“检索上下文不足时，模型用预训练知识脑补”。

输出用 0/1 更清晰：

- 1：答案完全可以从 context 推出
- 0：答案无法从 context 推出

In [6]:
faithfulness_prompt = PromptTemplate(
    input_variables=["question", "context", "generated_answer"],
    template="""
Question: {question}
Context: {context}
Generated Answer: {generated_answer}

Evaluate if the generated answer to the question can be deduced from the context.
Return score = 1 if the answer can be derived from the context, else score = 0.
You do NOT care whether the answer is correct in the real world.

Examples:

Question: What are the capitals of France and Spain?
Context: Paris is the capital of France and Madrid is the capital of Spain.
Generated Answer: Paris
Score should be 1.

Question: What are the capitals of France and Spain?
Context: London is the capital of France and Barcelona is the capital of Spain.
Generated Answer: London and Barcelona.
Score should be 1.

Question: What are the capitals of France and Spain?
Context: London is the capital of France and Madrid is the capital of Spain.
Generated Answer: Paris and Madrid.
Score should be 0 (uses pretrained knowledge not supported by context).

Question: What is the capital of France?
Context: Paris.
Generated Answer: Paris.
Score should be 0 (context doesn't state the relationship "capital of France").

Return your result as JSON.
""".strip(),
)

faithfulness_chain = faithfulness_prompt | llm.with_structured_output(ResultScore)


def evaluate_faithfulness(question: str, context: str, generated_answer: str) -> ResultScore:
    return faithfulness_chain.invoke(
        {"question": question, "context": context, "generated_answer": generated_answer}
    )

In [7]:
# quick test
question = "what is 3+3?"
context = "6"
generated_answer = "6"

result = evaluate_faithfulness(question, context, generated_answer)
result

ResultScore(score=1.0, explanation="The generated answer '6' can be directly deduced from the context '6', as it explicitly states the answer to the question 'what is 3+3?'. Therefore, the answer is supported by the context.")

## 3) Retrieval relevance：检索到的文档是否相关？

这类指标只评估 **retrieval**，不评估最终回答。

这里用一个简单近似：把多条 `contexts` 的相关度分别打分，再取平均。

- 0.00：完全无关
- 0.33：有点相关（提到关键词但不解答）
- 0.66：相关（能部分回答/强相关暗示）
- 1.00：高度相关（直接且完整回答问题）

In [8]:
def format_contexts(contexts: list[str]) -> str:
    return "\n\n".join([f"[Doc {i}] {c}" for i, c in enumerate(contexts)])


retrieval_relevance_prompt = PromptTemplate(
    input_variables=["question", "contexts"],
    template="""
Q: {question}
Docs:
{contexts}

Score each doc's relevance:
0.00 - Irrelevant: No relation to the question
0.33 - Somewhat relevant: Contains related keywords or concepts
0.66 - Relevant: Partially answers or strongly implies the answer
1.00 - Highly relevant: Directly and fully answers the question

Final score should be the average across all docs.
Return your result as JSON.
""".strip(),
)

retrieval_relevance_chain = retrieval_relevance_prompt | llm.with_structured_output(ResultScore)


def evaluate_retrieval_relevance(question: str, contexts: list[str]) -> ResultScore:
    return retrieval_relevance_chain.invoke(
        {"question": question, "contexts": format_contexts(contexts)}
    )

In [9]:
# quick test
question = "What is the capital of France?"
contexts = ["Paris is the capital of France.", "I was traveling in France last year."]

result = evaluate_retrieval_relevance(question, contexts)
result

ResultScore(score=1.0, explanation='Doc 0 directly states that Paris is the capital of France, providing a complete answer to the question. Doc 1 does not address the question at all, making it irrelevant. The average score is therefore 1.00.')